# Week 1 (continued) — Leakage-Safe Train/Val/Test Split

**Goal:** split the deduped DAIGT-V2 essays so that every score we report later is honest. The old version of this project hit 100% because its test essays were too similar to its training essays. Here, three rules prevent that:

1. **No near-duplicate cluster crosses a split** (`dup_cluster` from `02_dedup.ipynb` is a hard grouping constraint).
2. **Label ratio is identical** in train, val and test — stratified, not force-balanced.
3. **Two robustness slices are never trained on:** essays from two *held-out generator families* and from two *held-out prompts*. They measure whether the detector learned "AI writing" or just "the generators/topics it saw".

Output: `data/processed/splits.csv` (row id → split name) plus a data card. The essay text is **not** copied again; `src/data.py` joins the split onto `train_deduped.csv`.

In [1]:
import sys
sys.path.append('..')          # so `src` is importable when running from Notebooks/

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from src.data import load_deduped, load_splits

SEED = 42
pd.set_option('display.width', 200)

## Step 1 — Load, and check for formatting shortcuts

Before splitting, look for **surface artifacts**: properties of the raw string (not the writing) that differ between human and AI essays. A model will happily learn these instead of the real signal, which is exactly the trap the old 100% fell into.

Whitespace at the very start or end of the string is a candidate: some generation pipelines leave a leading space after the prompt. Count it per source on the *unstripped* file.

In [2]:
raw = pd.read_csv('../data/processed/train_deduped.csv')
ws = pd.DataFrame({
    'label':          raw['label'],
    'source':         raw['source'],
    'leading_space':  raw['text'].str.match(r'^\s'),
    'trailing_space': raw['text'].str.contains(r'\s$'),
})
print("% of essays, by label:")
print((ws.groupby('label')[['leading_space', 'trailing_space']].mean() * 100).round(1))
print("\n% of essays, by source (only sources where either is non-zero):")
by_src = (ws.groupby('source')[['leading_space', 'trailing_space']].mean() * 100).round(0)
print(by_src[(by_src > 0).any(axis=1)])

% of essays, by label:
       leading_space  trailing_space
label                               
0                0.0            50.1
1               21.7             0.8

% of essays, by source (only sources where either is non-zero):
                                    leading_space  trailing_space
source                                                           
NousResearch/Llama-2-7b-chat-hf             100.0             2.0
cohere-command                              100.0            21.0
darragh_claude_v6                            22.0             0.0
mistral7binstruct_v2                        100.0             0.0
mistralai/Mistral-7B-Instruct-v0.1          100.0             1.0
persuade_corpus                               0.0            53.0
radekgpt4                                     0.0            20.0


**Finding:** no human essay starts with whitespace, while several AI sources (`mistral7binstruct_v2`, `cohere-command`, both Llama-2-7b / Mistral-0.1 dumps) start *every* essay with one. Trailing whitespace goes the other way (common for human essays).

A word-level TF-IDF ignores this, but character n-grams and subword tokenizers (BERT, GPT-2 in later weeks) would see it, and "starts with a space → AI" would be a free, meaningless shortcut. So the shared loader `load_deduped()` **strips leading/trailing whitespace**. Every later notebook goes through the same loader, so they all get the same cleaned text.

In [3]:
df = load_deduped()            # same rows as `raw`, text stripped, index = row_id
assert len(df) == len(raw)
assert not df['text'].str.match(r'^\s').any() and not df['text'].str.contains(r'\s$').any()
assert (df['text'].str.split().str.len() == df['n_words']).all()   # stripping never changes word counts
print(df.shape)
df.head(3)

(44866, 7)


,text,label,prompt_name,source,RDizzl3_seven,n_words,dup_cluster
row_id,,,,,,,
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False,379,0
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False,366,1
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False,178,2


## Step 2 — The two open checks from `02_dedup`

### 2a. Does every prompt have both human and AI essays?

This matters for the held-out-prompt slice: a held-out prompt with only one label could not be scored as a detection task.

In [4]:
by_prompt = pd.crosstab(df['prompt_name'], df['label']).rename(columns={0: 'human', 1: 'ai'})
by_prompt['ai_share'] = (by_prompt['ai'] / by_prompt.sum(axis=1)).round(3)
by_prompt['RDizzl3_seven'] = df.groupby('prompt_name')['RDizzl3_seven'].first()
print("Every prompt has both labels:", (by_prompt[['human', 'ai']] > 0).all().all())
by_prompt.sort_values('ai_share')

Every prompt has both labels: True


label,human,ai,ai_share,RDizzl3_seven
prompt_name,,,,
Exploring Venus,1862,314,0.144,True
Driverless cars,1886,364,0.162,True
The Face on Mars,1583,310,0.164,True
Cell phones at school,1656,463,0.218,False
Grades for extracurricular activities,1626,490,0.232,False
Phones and driving,1168,415,0.262,False
Community service,1542,550,0.263,False
"""A Cowboy Who Rode the Waves""",1372,524,0.276,True
Facial action coding system,2167,917,0.297,True


**Yes, all 15 prompts have both labels.** But the AI share ranges from ~14% ("Exploring Venus") to ~70% ("Seeking multiple opinions"). So the *topic* alone carries information about the label: a model could learn "essays about distance learning are probably AI". The held-out-prompt slice measures how much the model leans on that. This goes in the data card.

### 2b. What are the cross-source and mixed-label clusters?

`02_dedup` found 1,069 clusters spanning more than one source and 2 clusters mixing human and AI. Look at what they are.

In [5]:
clusters = df.groupby('dup_cluster')
multi = df[df['dup_cluster'].map(clusters.size()) > 1]

source_combo = multi.groupby('dup_cluster')['source'].apply(lambda s: ' + '.join(sorted(set(s))))
print("Source combinations among multi-essay clusters:")
print(source_combo.value_counts())
print("\nClusters spanning more than one prompt:", (clusters['prompt_name'].nunique() > 1).sum())

Source combinations among multi-essay clusters:
source
persuade_corpus + train_essays            1067
persuade_corpus                              8
mistral7binstruct_v2 + persuade_corpus       2
Name: count, dtype: int64

Clusters spanning more than one prompt: 0


- **`persuade_corpus + train_essays` (1,067 clusters):** the same human essay appears in both. Kaggle's `train_essays` file was drawn from the Persuade corpus. Both copies are human, and grouping on `dup_cluster` already keeps them on the same side of the split, so this is harmless.
- **No cluster spans two prompts.** So holding out whole prompts can never cut a cluster in half.
- **`mistral7binstruct_v2 + persuade_corpus` (2 clusters):** these are the 2 mixed-label clusters. Read them.

In [6]:
mixed_ids = clusters['label'].nunique().loc[lambda s: s > 1].index
for cid in mixed_ids:
    pair = df[df['dup_cluster'] == cid].sort_values('label')
    human_words, ai_words = pair['text'].str.split().tolist()
    is_prefix = human_words[:len(ai_words)] == ai_words
    print(f"cluster {cid}: human {len(human_words)} words, 'AI' {len(ai_words)} words, "
          f"'AI' text is an exact prefix of the human text: {is_prefix}")
    print("   'AI' essay ends:  ...", ' '.join(ai_words[-15:]))
    print()

cluster 33338: human 497 words, 'AI' 413 words, 'AI' text is an exact prefix of the human text: True
   'AI' essay ends:  ... was very distracting. Clearly, students will not benefit if they are able to attend classes

cluster 36311: human 618 words, 'AI' 423 words, 'AI' text is an exact prefix of the human text: True
   'AI' essay ends:  ... attend online classes from home because by taking online classes, high school students can save



The "AI" essay in each pair is **word-for-word the opening of the human essay, cut off mid-sentence**. The generator was presumably given the human essay as context and echoed it back. There is no AI-written text in it, so the AI label is wrong.

**Decision:** drop these 2 rows, for the same reason the 2 sub-20-word essays were dropped in the EDA (broken generations, not real examples of either class). The human essays in those clusters stay.

In [7]:
bad_rows = df.index[df['dup_cluster'].isin(mixed_ids) & (df['label'] == 1)]
assert len(bad_rows) == 2
df = df.drop(index=bad_rows)
assert (df.groupby('dup_cluster')['label'].nunique() == 1).all()   # every cluster is now single-label
print("Dropped row_ids:", list(bad_rows), "| remaining:", len(df))

Dropped row_ids: [33338, 36311] | remaining: 44864


## Step 3 — Carve out the two robustness slices

A normal test set only shows how well the model does on *more of the same*. A real detector meets new generators and new topics, so we hold two slices out of training entirely:

| Slice | What's held out | Why these |
|---|---|---|
| `heldout_prompt` | every essay (human and AI) on **"Does the electoral college work?"** and **"Summer projects"** | Combined AI share ≈ 37%, close to the overall 39%, so scores are comparable with the main test set. One is a Kaggle-test (`RDizzl3_seven`) prompt, one is not. |
| `heldout_generator` | AI essays from **Claude** (`darragh_claude_v6`, `_v7`) and **Falcon-180B** (`falcon_180b_v1`) | Whole *families*: no other Claude or Falcon version stays in training, so the test is truly unseen. (Holding out Mistral v2 while v1 trains would be a weak test.) |

**Order matters.** The prompt slice is taken first, so Claude/Falcon essays on the held-out prompts belong to the prompt slice. The generator slice is then the remaining Claude/Falcon essays. Neither family appears in train/val/test at all.

The generator slice is **AI-only**, so on its own it is scored by *detection rate* (recall: the % of its essays flagged as AI). For AUC/F1 it can be paired with the human essays of the main test set.

*Note for Week 5:* generator attribution needs Claude and Falcon in its training data, so the attribution model will get its own split. These held-out slices are for the human-vs-AI detector only.

In [8]:
HELDOUT_PROMPTS = ['Does the electoral college work?', 'Summer projects']
HELDOUT_GENERATORS = ['darragh_claude_v6', 'darragh_claude_v7', 'falcon_180b_v1']
assert set(HELDOUT_PROMPTS) <= set(df['prompt_name']) and set(HELDOUT_GENERATORS) <= set(df['source'])

df['split'] = None
in_prompt = df['prompt_name'].isin(HELDOUT_PROMPTS)
df.loc[in_prompt, 'split'] = 'heldout_prompt'
df.loc[~in_prompt & df['source'].isin(HELDOUT_GENERATORS), 'split'] = 'heldout_generator'

pool = df[df['split'].isna()]          # what's left for train/val/test
print(df['split'].value_counts(dropna=False))
print(f"\nPool for train/val/test: {len(pool)} essays, AI share {pool['label'].mean():.3f}")

split
None                 35008
heldout_prompt        7135
heldout_generator     2721
Name: count, dtype: int64

Pool for train/val/test: 35008 essays, AI share 0.346


**Note the pool's AI share: 34.6%, not 39%.** The generator slice removed 2,721 essays, all AI, so fewer AI essays are left. We keep this natural ratio rather than discarding human essays to restore 39% (no force-balancing). Stratification then makes train, val and test all match it.

## Step 4 — Split the pool 80/10/10, grouped and stratified

**Tool: `StratifiedGroupKFold`** (scikit-learn). It cuts the data into *k* folds such that:
- every `dup_cluster` (the *group*) lands entirely inside one fold — the hard constraint;
- each fold's label ratio is as close as possible to the overall ratio — the stratification.

With `k = 10`, each fold is ~10% of the pool: **fold 0 → test, fold 1 → val, folds 2–9 → train**. That gives 80/10/10 from a single pass, with no second split to reason about.

**Sanity check first, on a toy case** (same habit as `02_dedup`): 12 rows in 6 groups, where group A has 3 rows. If the tool works, no group should ever have rows in two folds.

In [9]:
toy = pd.DataFrame({
    'group': list('AAABBCCDDEEF'),
    'label': [1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1],
})
toy_fold = np.empty(len(toy), dtype=int)
for k, (_, idx) in enumerate(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
                             .split(toy, toy['label'], groups=toy['group'])):
    toy_fold[idx] = k
toy['fold'] = toy_fold
print(toy.groupby('group')['fold'].unique())
assert (toy.groupby('group')['fold'].nunique() == 1).all()
print("\nToy check passed: every group sits in exactly one fold.")

group
A    [0]
B    [1]
C    [2]
D    [0]
E    [2]
F    [1]
Name: fold, dtype: object

Toy check passed: every group sits in exactly one fold.


Every group stays whole. Now the real pool. `shuffle=True` with a fixed `random_state` makes the split random but reproducible: re-running the notebook gives the identical split.

In [10]:
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
fold = pd.Series(-1, index=pool.index)
for k, (_, idx) in enumerate(sgkf.split(pool, pool['label'], groups=pool['dup_cluster'])):
    fold.iloc[idx] = k
assert (fold >= 0).all()

df.loc[pool.index, 'split'] = np.select([fold == 0, fold == 1], ['test', 'val'], default='train')
assert df['split'].notna().all()
df['split'].value_counts()

split
train                28006
heldout_prompt        7135
test                  3501
val                   3501
heldout_generator     2721
Name: count, dtype: int64

## Step 5 — Verify the split

Each check below is an `assert`, so the notebook stops loudly if anything is wrong instead of saving a bad split.

**5a. Leakage:** no cluster in two splits, and the held-out prompts/generators never appear in train/val/test.

In [11]:
assert (df.groupby('dup_cluster')['split'].nunique() == 1).all(), "a cluster crosses splits!"

main = df[df['split'].isin(['train', 'val', 'test'])]
assert not main['prompt_name'].isin(HELDOUT_PROMPTS).any()
assert not main['source'].isin(HELDOUT_GENERATORS).any()
assert (df.loc[df['split'] == 'heldout_generator', 'label'] == 1).all()

# multi-essay clusters, by split (the near-duplicates now live together)
multi_ids = clusters.size().loc[lambda s: s > 1].index
print("Multi-essay clusters per split:")
print(df[df['dup_cluster'].isin(multi_ids)].groupby('split')['dup_cluster'].nunique())
print("\nAll leakage checks passed.")

Multi-essay clusters per split:
split
heldout_prompt    525
test               52
train             444
val                56
Name: dup_cluster, dtype: int64

All leakage checks passed.


**5b. Size and label ratio per split.** Train, val and test should have the same AI share as the pool they came from.

In [12]:
order = ['train', 'val', 'test', 'heldout_prompt', 'heldout_generator']
summary = df.groupby('split').agg(essays=('label', 'size'), ai=('label', 'sum'), ai_share=('label', 'mean'))
summary['human'] = summary['essays'] - summary['ai']
summary['% of data'] = (summary['essays'] / len(df) * 100).round(1)
summary = summary.loc[order, ['essays', '% of data', 'human', 'ai', 'ai_share']].round(3)
summary

,essays,% of data,human,ai,ai_share
split,,,,,
train,28006,62.4,18325,9681,0.346
val,3501,7.8,2291,1210,0.346
test,3501,7.8,2291,1210,0.346
heldout_prompt,7135,15.9,4464,2671,0.374
heldout_generator,2721,6.1,0,2721,1.000


**5c. Do sources and prompts spread evenly across train/val/test?** Grouping by cluster could in principle push one source into one split. Each row below should be roughly 80/10/10.

In [13]:
pct = lambda col: (pd.crosstab(main[col], main['split'], normalize='index')[['train', 'val', 'test']] * 100).round(1)
print("% of each SOURCE's essays in each split")
print(pct('source'))
print("\n% of each PROMPT's essays in each split")
print(pct('prompt_name'))

% of each SOURCE's essays in each split
split                               train   val  test
source                                               
NousResearch/Llama-2-7b-chat-hf      81.5  11.5   7.0
chat_gpt_moth                        79.8   9.7  10.5
cohere-command                       81.0   9.3   9.7
kingki19_palm                        77.7  10.8  11.5
llama2_chat                          80.6  10.5   8.9
llama_70b_v1                         81.4   9.3   9.4
mistral7binstruct_v1                 79.1  10.5  10.4
mistral7binstruct_v2                 79.9   9.7  10.4
mistralai/Mistral-7B-Instruct-v0.1   82.4   7.0  10.6
palm-text-bison1                     82.3   8.7   9.0
persuade_corpus                      80.0  10.0  10.0
radek_500                            77.6  11.2  11.2
radekgpt4                            85.0   7.0   8.0
train_essays                         81.2  10.0   8.8

% of each PROMPT's essays in each split
split                                  train   val  tes

**5d. Essay length by split and label.** The EDA found AI essays are shorter and more uniform than human ones. That is a real but *surface* signal. Here we only check that it looks the same in every split, so no split is easier or harder by accident.

In [14]:
df.groupby(['split', 'label'])['n_words'].agg(['mean', 'std', 'median']).round(0).loc[order]

mean    std  median
split             label                      
train             0      403.0  184.0   367.0
                  1      326.0   95.0   333.0
val               0      394.0  175.0   363.0
                  1      329.0   92.0   336.0
test              0      406.0  185.0   370.0
                  1      327.0   96.0   339.0
heldout_prompt    0      501.0  199.0   474.0
                  1      367.0  113.0   348.0
heldout_generator 1      304.0   51.0   303.0

**5e. Read a few real examples** from the held-out slices, so the slices are concrete rather than just counts.

In [15]:
for split in ['heldout_prompt', 'heldout_generator']:
    print(f"===== {split} =====")
    for _, r in df[df['split'] == split].sample(2, random_state=SEED).iterrows():
        print(f"[label={r['label']} | {r['source']} | {r['prompt_name']} | {r['n_words']} words]")
        print(r['text'][:400].replace('\n', ' '), '...\n')

===== heldout_prompt =====
[label=0 | persuade_corpus | Does the electoral college work? | 490 words]
Dear Mr. Senator,  As you are most likely aware, the Electoral College is a long-debated, controversial process. I believe that it is in the best interest for the citizens of America that the Electoral College be abolished. Under the Electoral College system, voters do not vote for the president but rather for a slate of electors who will vote for the president. Voters cannot control who the elect ...

[label=1 | NousResearch/Llama-2-7b-chat-hf | Does the electoral college work? | 300 words]
Dear State Senator,  I hope this letter finds you in good health and high spirits. As a concerned citizen, I am writing to express my opinion on the Electoral College system and its role in electing the President of the United States. I strongly believe that the Electoral College is a vital component of our democratic system and should be retained or changed to election by popular vote.  One of th 

## Step 6 — Save

Only the assignment is saved: `row_id → split`, a few hundred KB. The text stays in `train_deduped.csv`, which is already committed and close to GitHub's 100 MB file limit, so writing four more copies of it would be a problem. The 2 dropped rows simply have no entry in `splits.csv`.

Then reload via `src.data.load_splits()`, the way every later notebook will, and confirm it matches what was built here.

In [16]:
df[['split']].to_csv('../data/processed/splits.csv')        # index is row_id

reloaded = load_splits()
assert reloaded.index.equals(df.index) and (reloaded['split'] == df['split']).all()
assert reloaded['text'].equals(df['text'])
print("Saved and verified:", reloaded.shape)
print(reloaded['split'].value_counts())

# the way later notebooks will use it:
train = load_splits('train')
print("\nload_splits('train'):", train.shape)

Saved and verified: (44864, 8)
split
train                28006
heldout_prompt        7135
test                  3501
val                   3501
heldout_generator     2721
Name: count, dtype: int64



load_splits('train'): (28006, 8)


## Key takeaways for the baseline notebook

- **Load data with `from src.data import load_splits`**: `load_splits('train')`, `load_splits('val', 'test')`, etc. Don't read the CSVs directly. The loader applies the whitespace strip, and skipping it would reintroduce the leading-space shortcut.
- **Splits:** train 28,006 / val 3,501 / test 3,501 (80/10/10 of the pool), each exactly 34.6% AI; held-out prompt 7,135 (37.4% AI); held-out generator 2,721 (all AI). No near-duplicate cluster crosses any boundary.
- **Why 65/35 and not 61/39:** the generator slice removes 2,721 essays from the pool, all of them AI, so the pool's AI share drops from 39% to 34.6%. Stratification keeps that ratio identical across train/val/test. Restoring 39% would mean discarding human essays, i.e. force-balancing, which we decided against. Report precision/recall/F1/AUC, not just accuracy: a model that always says "human" already gets 65%.
- **Report three numbers, not one:** the main test set, the held-out-prompt slice (unseen topics), and the held-out-generator slice (unseen families: Claude, Falcon-180B; AI-only, so detection rate, or AUC when paired with test-set humans). The gap between them is the honest story.
- **Tune only on `val`.** `test` and both held-out slices are touched once, at the end.
- **Known surface signals** (in the data card): AI essays are shorter and more uniform; AI share varies 14–70% by prompt; whitespace artifacts (now neutralised). A strong baseline score should be checked against a length-only or prompt-only model to see how much of it these signals explain.
- **Generator attribution (Week 5)** needs Claude and Falcon in training, so it gets its own split; these held-out slices are for the human-vs-AI detector.